# Importación masiva de candidatos con IA

Documentación de cómo funciona la carga de hojas de vida en **Pander**: qué hace el modelo, cómo se
conecta el frontend con el backend, qué responde la API y cómo probarla.

**Contenido**

1. [¿Qué hace?](#1.-¿Qué-hace?)
2. [Dónde está cada cosa](#2.-Dónde-está-cada-cosa)
3. [El modelo](#3.-El-modelo)
4. [Formatos que se pueden cargar](#4.-Formatos-que-se-pueden-cargar)
5. [Reglas de negocio](#5.-Reglas-de-negocio)
6. [La API](#6.-La-API)
7. [Probar la API paso a paso](#7.-Probar-la-API-paso-a-paso)
8. [La importación por Excel sigue igual](#8.-La-importación-por-Excel-sigue-igual)
9. [Cómo lo usa el frontend](#9.-Cómo-lo-usa-el-frontend)
10. [Configuración](#10.-Configuración)
11. [Pruebas](#11.-Pruebas)
12. [Reentrenar el modelo](#12.-Reentrenar-el-modelo)
13. [Problemas frecuentes](#13.-Problemas-frecuentes)

## 1. ¿Qué hace?

En **Importación Masiva → Candidatos** ahora hay dos formas de cargar candidatos, y las dos conviven:

| Origen | Para qué sirve | Qué se sube |
|---|---|---|
| **Matriz de Excel** (como siempre) | Cuando ya tienes los datos en una tabla | Un `.xlsx` con la plantilla |
| **Hojas de vida (IA)** | Cuando solo tienes los CV | Uno o **varios** archivos: PDF, DOCX, PNG, JPG, HTML, MD, TXT (se pueden mezclar) |

El flujo con hojas de vida es siempre en dos pasos, igual que el Excel:

```
 Archivos  ──►  1. ANALIZAR (dry_run)  ──►  revisar lo extraído  ──►  2. IMPORTAR
                no guarda nada                datos + avisos            crea Candidato, sus datos
                                                                         y guarda el CV como anexo
```

Para cada archivo el backend hace esto:

1. **Valida** el archivo: extensión permitida, no vacío, tamaño máximo y que el contenido sea de verdad de ese tipo.
2. **Lee el texto** (PDF/DOCX/HTML/MD/TXT directo; imágenes y PDF escaneados con OCR).
3. **Extrae los datos** con el modelo BiLSTM-CRF: nombres, apellidos, tipo y número de documento, correo y teléfono.
4. **Mapea y valida** contra los campos de `DatosCandidato` (el **email es obligatorio**).
5. **Revisa duplicados** en la base de datos y dentro del mismo lote.
6. Si no es `dry_run`: guarda el candidato y el archivo original.

## 2. Dónde está cada cosa

```
pander_back/
├── ai/                                   ← este directorio
│   ├── entrenar_colab.ipynb              entrenar el modelo en Google Colab
│   └── documentacion_importacion_ia.ipynb   este documento
├── config/settings.py                    ajustes CV_* (sección 10)
└── apps/candidatos/
    ├── views_importacion.py              importación por Excel (+ plantillas)
    ├── views_importacion_cv.py           importación por hojas de vida  ← la API de este documento
    ├── urls_importacion.py               rutas  /api/importacion/...
    ├── ia/
    │   ├── ner_documentos.py             el modelo (BiLSTM-CRF) y su CLI: gen / train / predict
    │   ├── modelo/modelo.pt              el modelo ya entrenado
    │   ├── lectores.py                   texto de PDF, DOCX, imágenes, HTML, MD, TXT
    │   ├── extractor.py                  carga el modelo una sola vez y extrae
    │   ├── mapeo.py                      salida del modelo → campos de DatosCandidato + validaciones
    │   └── servicio.py                   validar archivos, analizar lotes, importar
    └── pruebas/                          pruebas unitarias y de integración

Frontend/src/
├── pages/ImportacionMasiva/
│   ├── ImportacionMasiva.jsx             página (selector Excel / Hojas de vida)
│   ├── ImportacionCV.jsx                 pantalla de carga de hojas de vida
│   ├── cvImport.js                       lógica: lotes, validación de archivos, duplicados
│   └── cvImport.test.js · ImportacionCV.test.jsx
└── services/index.js                     importacionService (consumo de la API)
```

## 3. El modelo

Es una red recurrente **BiLSTM-CRF** hecha en PyTorch que etiqueta cada palabra del texto
(*Named Entity Recognition*):

```
palabra ─► embedding de palabra ─┐
letras  ─► embedding + CNN       ├─► concatena ─► BiLSTM (2 capas, 128) ─► capa lineal ─► CRF ─► etiqueta por palabra
forma   ─► embedding             ┘                                                          (B-/I-/O)
```

- El **embedding de letras (CNN)** ayuda con nombres y correos que el modelo nunca vio.
- La **forma** de la palabra (mayúscula, dígitos, símbolos) ayuda a distinguir un nombre de un número.
- El **CRF** del final obliga a que las etiquetas tengan sentido en secuencia (no empieza un apellido a mitad de un correo).

Entidades que reconoce (formato BIO: `B-` inicio, `I-` continuación, `O` nada):

| Etiqueta | Campo de salida |
|---|---|
| `PRIMER_NOMBRE`, `SEGUNDO_NOMBRE` | `primer_nombre`, `segundo_nombre` |
| `PRIMER_APELLIDO`, `SEGUNDO_APELLIDO` | `primer_apellido`, `segundo_apellido` |
| `TIPO_DOC`, `NUM_DOC` | `tipo_doc`, `num_doc` |
| `EMAIL` | `email` |
| `TELEFONO` | `telefono` |

**Con qué se entrena:** documentos sintéticos (Faker `es_CO` con plantillas desordenadas, `--synth`) y, opcionalmente,
hojas de vida reales anotadas en JSONL (`--real`), que pesan más (`--real_weight`, por defecto 3 veces).
Se entrena en Colab con `entrenar_colab.ipynb` (sección 12).

Formato de una línea del JSONL de datos reales:

```json
{"tokens": ["Juan", "Pérez", "juan@x.com"], "tags": ["B-PRIMER_NOMBRE", "B-PRIMER_APELLIDO", "B-EMAIL"]}
```

> ⚠️ `ner_documentos.py` **no se debe modificar** sin reentrenar: el modelo guardado depende de su forma de
> partir el texto en palabras y de la clase `BiLSTMCRF`.

## 4. Formatos que se pueden cargar

| Formato | Cómo se lee | Necesita OCR |
|---|---|---|
| **PDF** con texto | `pdfplumber` (respaldo: `markitdown`) | No |
| **PDF** escaneado | Se detecta porque casi no trae texto → se pasa a imagen | **Sí** (`easyocr`) |
| **DOCX** | `markitdown`; respaldo propio que lee el XML (incluye encabezados y pies) | No |
| **PNG / JPG / JPEG** | OCR | **Sí** (`easyocr`) |
| **HTML / HTM** | `markitdown`; respaldo con el parser de Python (ignora scripts y estilos) | No |
| **MD / TXT** | Texto plano (UTF-8, con o sin BOM; si falla, Windows-1252) | No |

Controles de seguridad al recibir cada archivo: se usa solo el nombre base (sin rutas), se comprueba la
**firma** del contenido (un `.pdf` debe empezar por `%PDF`, un `.png` por su cabecera, un `.docx` debe ser un
zip con `word/document.xml`, los de texto no pueden tener bytes nulos) y se guarda con un nombre generado, nunca
con el que mandó el usuario.

## 5. Reglas de negocio

Las mismas reglas aplican al Excel y a las hojas de vida (código en `ia/mapeo.py` y `views_importacion.py`).

| Campo | Regla |
|---|---|
| `primer_nombre`, `primer_apellido` | **Obligatorios**. No pueden traer dígitos ni `@` (no son nombres). |
| `email` | **Obligatorio** y con formato válido. Se guarda en minúsculas y sin espacios. |
| `segundo_nombre`, `segundo_apellido`, `telefono` | Opcionales |
| `tipo_documento`, `numero_documento` | Opcionales. El tipo se busca en el catálogo (acepta "CC", "C.C.", "Cédula de Ciudadanía"...). Si no está en el catálogo, se importa sin tipo y se avisa. |

Otras reglas:

- **Longitudes**: las de las columnas de la BD. Si un campo opcional se pasa, se omite con un aviso; si es obligatorio, es un error.
- **Duplicados**: un candidato ya existe en la compañía si coincide el **correo** (sin importar mayúsculas) o el
  **tipo + número de documento**. También se detectan dos archivos del mismo lote que sean la misma persona
  (el primero se queda, el otro queda con error). El mismo correo en otra compañía sí se permite.
- **Avisos** (no impiden importar): hay número pero no tipo de documento, el tipo no está en el catálogo, se detectó
  más de una persona en el documento, el nombre salió del título del documento.
- Cada archivo es independiente: **un archivo con error no afecta a los demás**.
- Al importar se guarda el **CV original** como anexo del candidato (`AnexoCandidato`) en
  `MEDIA_ROOT/candidatos/<compañía>/<candidato>/<uuid>.<ext>`.

## 6. La API

No exige autenticación propia (igual que el resto de `/api/importacion/`). Base: `http://127.0.0.1:8000`.

### `GET /api/importacion/cv/estado/`

Dice si el modelo está listo y qué límites hay. El frontend lo consulta al abrir la pantalla.

```json
{"disponible": true, "motivo": null, "version_modelo": "v5-lectura-pdfplumber-persona-unica",
 "formatos": ["docx","htm","html","jpeg","jpg","md","pdf","png","txt"], "max_archivos": 50, "max_mb": 10}
```

### `POST /api/importacion/companias/<compania>/candidatos/cv/`  (`multipart/form-data`)

| Campo | Obligatorio | Descripción |
|---|---|---|
| `archivos` | sí | Los archivos. El campo se **repite** una vez por archivo (también se acepta `archivo`). |
| `dry_run` | no (`false`) | `true` → solo analiza y devuelve lo extraído, **no guarda nada** |
| `omitir_errores` | no (`true`) | `true` → importa los válidos y reporta los demás · `false` → si hay algún error no se importa nada |
| `usuario_id` | no (`1`) | Id del usuario que importa (auditoría) |

**Respuesta** — mismo contrato que el Excel: `modo`, `total`, `ok`/`creados`, `errores` y un `log` con una entrada por archivo.
`fila` es la posición del archivo **dentro de esa petición** (empieza en 1).

```json
{
  "modo": "validacion", "total": 2, "ok": 1, "errores": 1,
  "log": [
    {"fila": 1, "archivo": "sofia.txt", "estado": "OK", "mensaje": "Sofía Ruiz <sofia.ruiz@example.com>",
     "datos": {"primer_nombre": "Sofía", "segundo_nombre": "Alejandra", "primer_apellido": "Ruiz",
               "segundo_apellido": "Castro", "tipo_documento": "Cédula de Ciudadanía",
               "numero_documento": "1030456789", "email": "sofia.ruiz@example.com", "telefono": "3101234567"},
     "advertencias": []},
    {"fila": 2, "archivo": "otro.pdf", "estado": "ERROR",
     "mensaje": "No se encontró un correo electrónico (campo obligatorio).", "datos": {"...": "..."}, "advertencias": []}
  ]
}
```

Al importar (`dry_run=false`) la respuesta trae `modo: "importacion"`, `creados`, y cada entrada importada
agrega `candidato_id`.

**Códigos de respuesta**

| Código | Cuándo |
|---|---|
| `200` | Análisis (`dry_run=true`) correcto |
| `201` | Importación con al menos un candidato creado |
| `400` | No se mandaron archivos, o más de `max_archivos` en una petición |
| `404` | La compañía no existe |
| `422` | No se importó ninguno, o hay errores con `omitir_errores=false`. Trae `error` y el `log` por archivo |
| `503` | El modelo no está disponible (falta `modelo.pt`, `torch` o `pytorch-crf`). Trae `error` |

## 7. Probar la API paso a paso

Las celdas siguientes llaman a la API de verdad. Antes de ejecutarlas:

1. Deja el servidor corriendo (`python manage.py runserver`).
2. Ajusta `COMPANIA_ID` a una compañía que exista en tu base de datos.
3. `IMPORTAR` está en `False`: solo se **analiza**, no se guarda nada. Cámbialo a `True` si quieres crear los candidatos.

Se necesita `requests` (`pip install requests`).

In [1]:
import json
import os
import requests

BASE_URL = os.environ.get("PANDER_API", "http://127.0.0.1:8000")
COMPANIA_ID = int(os.environ.get("PANDER_COMPANIA", "1"))
IMPORTAR = False      # True = guarda de verdad los candidatos

# 1) ¿está listo el modelo?
estado = requests.get(f"{BASE_URL}/api/importacion/cv/estado/", timeout=30).json()
print(json.dumps(estado, indent=2, ensure_ascii=False))

{
  "disponible": true,
  "motivo": null,
  "version_modelo": "v5-lectura-pdfplumber-persona-unica",
  "formatos": [
    "docx",
    "htm",
    "html",
    "jpeg",
    "jpg",
    "md",
    "pdf",
    "png",
    "txt"
  ],
  "max_archivos": 50,
  "max_mb": 10
}


Tres hojas de vida de ejemplo en tres formatos distintos. La tercera **no trae correo** a propósito, para ver el error.

In [2]:
EJEMPLOS = {
    "sofia.txt": ("text/plain", """Sofía Alejandra Ruiz Castro
Diseñadora Gráfica

Datos de contacto
Correo electrónico: sofia.ruiz@example.com
Teléfono: 3101234567
Cédula de Ciudadanía: 1030456789

Perfil
Diseñadora con 5 años de experiencia en identidad visual.
"""),
    "andres.md": ("text/markdown", """# Andrés Felipe Torres Díaz

- Email: andres.torres@example.org
- Celular: 3209998877
- Pasaporte: AB123456

## Experiencia
Desarrollador web en Acme (2020 - 2024).
"""),
    "sin_correo.html": ("text/html", """<html><body>
<h1>Camila Andrea Vargas Peña</h1>
<p>Teléfono: 3157778899</p><p>Cédula de Ciudadanía: 52123456</p><p>Contadora pública.</p>
</body></html>"""),
}

def enviar(archivos, dry_run=True, omitir_errores=True):
    """archivos: {nombre: (mime, contenido_str_o_bytes)}  →  respuesta de la API"""
    partes = [("archivos", (nombre, contenido if isinstance(contenido, bytes) else contenido.encode("utf-8"), mime))
              for nombre, (mime, contenido) in archivos.items()]
    datos = {"dry_run": str(dry_run).lower(), "omitir_errores": str(omitir_errores).lower()}
    return requests.post(f"{BASE_URL}/api/importacion/companias/{COMPANIA_ID}/candidatos/cv/",
                         files=partes, data=datos, timeout=600)

# 2) analizar (dry_run=true): no guarda nada
r = enviar(EJEMPLOS, dry_run=True)
print("HTTP", r.status_code)
cuerpo = r.json()
print(f"total={cuerpo['total']}  ok={cuerpo['ok']}  errores={cuerpo['errores']}\n")
for e in cuerpo["log"]:
    print(f"fila {e['fila']}  {e['archivo']:<16} {e['estado']:<6} {e['mensaje']}")

HTTP 200
total=3  ok=2  errores=1

fila 1  sofia.txt        OK     Sofía Ruiz <sofia.ruiz@example.com>
fila 2  andres.md        OK     Andrés Torres <andres.torres@example.org>
fila 3  sin_correo.html  ERROR  No se encontró un correo electrónico (campo obligatorio).


Una entrada completa del `log`, para ver exactamente qué recibe el frontend (aquí el archivo con error: aunque no se
pueda importar, igual se devuelve lo que la IA alcanzó a leer, para que el usuario lo revise):

In [3]:
print(json.dumps(cuerpo["log"][2], indent=2, ensure_ascii=False))

{
  "fila": 3,
  "archivo": "sin_correo.html",
  "estado": "ERROR",
  "mensaje": "No se encontró un correo electrónico (campo obligatorio).",
  "datos": {
    "primer_nombre": "Camila",
    "segundo_nombre": "Andrea",
    "primer_apellido": "Vargas",
    "segundo_apellido": "Peña",
    "tipo_documento": "Cédula de Ciudadanía",
    "numero_documento": "52123456",
    "email": null,
    "telefono": "3157778899"
  },
  "advertencias": []
}


### Importar de verdad

Con `omitir_errores=true` (lo normal) se importan las hojas de vida válidas y las demás quedan reportadas. Esta celda solo
hace algo si pones `IMPORTAR = True` arriba.

In [4]:
if IMPORTAR:
    r = enviar(EJEMPLOS, dry_run=False, omitir_errores=True)
    print("HTTP", r.status_code)                  # 201 si se creó al menos uno
    cuerpo = r.json()
    print(f"creados={cuerpo['creados']}  errores={cuerpo['errores']}")
    for e in cuerpo["log"]:
        print(e["archivo"], "→ candidato", e.get("candidato_id"), "|", e["mensaje"])
else:
    print("IMPORTAR = False: no se guardó nada.")

IMPORTAR = False: no se guardó nada.


### Cuando algo sale mal

La API responde con un código y un `error` legible. Tres casos típicos:

In [5]:
url = f"{BASE_URL}/api/importacion/companias/{COMPANIA_ID}/candidatos/cv/"

# a) sin archivos → 400
r = requests.post(url, data={"dry_run": "true"}, timeout=30)
print("sin archivos        :", r.status_code, r.json())

# b) compañía que no existe → 404
r = requests.post(f"{BASE_URL}/api/importacion/companias/999999/candidatos/cv/",
                  files=[("archivos", ("a.txt", b"hola", "text/plain"))], data={"dry_run": "true"}, timeout=30)
print("compañía inexistente:", r.status_code, r.json())

# c) formato no permitido: la petición va bien, pero ese archivo queda con ERROR en el log
r = enviar({"programa.exe": ("application/octet-stream", b"MZ\x90\x00")})
print("formato no permitido:", r.status_code, r.json()["log"][0]["mensaje"])

sin archivos        : 400 {'error': "Se requiere al menos un archivo (campo 'archivos')."}
compañía inexistente: 404 {'error': 'La compañía indicada no existe.'}
formato no permitido: 200 Formato '.exe' no permitido. Use: docx, htm, html, jpeg, jpg, md, pdf, png, txt.


## 8. La importación por Excel sigue igual

La matriz de Excel no cambió su forma de trabajar (plantilla, validar, importar). Lo único nuevo es que, en
**Candidatos**, `email` ahora es **obligatorio**: la columna debe existir y cada fila debe traer un correo válido.

| Entidad | Ruta |
|---|---|
| Plantilla | `GET /api/importacion/plantilla/<entidad>/` |
| Candidatos (Excel) | `POST /api/importacion/companias/<compania>/candidatos/` con el campo `archivo` y `dry_run` |

Ejemplo: así se ve la cabecera de la plantilla de candidatos (requiere `openpyxl`):

In [6]:
import io

try:
    import openpyxl
    r = requests.get(f"{BASE_URL}/api/importacion/plantilla/candidatos/", timeout=30)
    ws = openpyxl.load_workbook(io.BytesIO(r.content)).active
    print([celda.value for celda in ws[1]])
except ImportError:
    print("Instala openpyxl para ver la plantilla:  pip install openpyxl")

['primer_nombre', 'segundo_nombre', 'primer_apellido', 'segundo_apellido', 'tipo_documento', 'numero_documento', 'email', 'telefono']


## 9. Cómo lo usa el frontend

La página `ImportacionMasiva.jsx` muestra, solo en **Candidatos**, un selector *Matriz de Excel* / *Hojas de vida (IA)*.
El segundo modo es el componente `ImportacionCV.jsx`, que usa la lógica de `cvImport.js` y el servicio `importacionService`
(`services/index.js`).

1. **Al abrir** consulta `GET /cv/estado/`. Si el modelo no está disponible, avisa el motivo y bloquea el botón *Analizar*.
2. **Al seleccionar o arrastrar archivos** (varios a la vez, formatos mezclados) se filtran en el navegador:
   formato permitido, no vacío, tamaño máximo (el que informa el backend), sin repetidos y máximo 500 por carga.
   Los rechazados se listan con el motivo.
3. **Analizar** manda los archivos en **lotes de 10** (`dry_run=true`) y muestra una barra de progreso. Cada archivo queda
   *Válido* o *Error*, con los datos extraídos y los avisos. Los duplicados **entre lotes distintos** los marca el
   frontend (dentro de un mismo lote ya los marca el backend).
4. **Importar** manda solo los archivos válidos, otra vez por lotes (`dry_run=false`). Los resultados quedan *Importado* o *Error*.
   El análisis no se repite en el modelo: el backend guarda en caché el resultado de cada archivo (por hash del contenido) durante una hora.
5. La casilla **«Importar solo las válidas»** (activa por defecto) se manda como `omitir_errores`. Si se desactiva y hay
   errores, el botón *Importar* queda bloqueado hasta que se quiten esos archivos.
6. **Cancelar** corta el proceso entre lotes y conserva lo ya procesado.

Detalles del consumo (`services/index.js`):

```js
importacionService.candidatosCV(cid, formData, config)
// POST /api/importacion/companias/<cid>/candidatos/cv/
// - Content-Type multipart/form-data (el axios del proyecto trae application/json por defecto)
// - timeout de 10 minutos: el OCR y el modelo pueden tardar bastante más que los 15 s por defecto
```

Si un lote falla por red o por un 5xx, solo ese lote queda con error y el resto continúa. Si el servidor responde
`503` (modelo apagado), `404` o `400`, se detiene todo y se muestra el mensaje.

## 10. Configuración

En `config/settings.py` (todas se pueden sobrescribir con variables de entorno):

| Ajuste | Por defecto | Para qué |
|---|---|---|
| `CV_MODEL_PATH` | `apps/candidatos/ia/modelo/modelo.pt` | Dónde está el modelo entrenado |
| `CV_DEVICE` | automático | `"cpu"` o `"cuda"`; vacío = usa GPU si hay |
| `CV_MAX_ARCHIVOS` | `50` | Archivos por petición |
| `CV_MAX_BYTES` | `10 MB` | Tamaño máximo por archivo |
| `CV_MAX_PAGINAS` | `10` | Páginas que se leen de cada PDF |
| `CV_CACHE_SEGUNDOS` | `3600` | Cuánto se recuerda el resultado de un archivo ya procesado |

Dependencias (ya están en `requirements.txt`): `torch`, `pytorch-crf`, `pdfplumber`, `markitdown`, `openpyxl`, `pillow`
y, solo para imágenes y PDF escaneados, `easyocr`.

> En producción conviene subir `CV_MAX_BYTES`/`CV_MAX_ARCHIVOS` solo si el servidor aguanta: cada archivo se procesa
> uno a la vez (el modelo y el OCR no son seguros en paralelo dentro del mismo proceso).

## 11. Pruebas

**Backend** (Django, desde `pander_back/`):

```bash
python manage.py test apps.candidatos.pruebas
```

| Archivo | Qué prueba |
|---|---|
| `test_mapeo.py` | Reglas de negocio: email obligatorio y válido, tipos de documento, longitudes, avisos |
| `test_lectores.py` | Lectura de cada formato (TXT, MD, HTML, DOCX, PDF, imágenes) |
| `test_extractor.py` | Disponibilidad del modelo y extracción con el **modelo real** |
| `test_importacion_cv_api.py` | API de hojas de vida de punta a punta: validaciones, duplicados, caché, importación parcial, rollback, carga masiva multi-formato |
| `test_importacion_excel.py` | El Excel sigue funcionando y exige `email` |
| `test_contrato_frontend.py` | El formato exacto de respuesta del que depende la pantalla React |

Las pruebas de imágenes con OCR se omiten solas si `easyocr` no está instalado.

**Frontend** (desde `Frontend/`):

```bash
node --test src/pages/ImportacionMasiva/cvImport.test.js      # lógica pura, sin instalar nada
npx vitest run --environment jsdom src/pages/ImportacionMasiva/ImportacionCV.test.jsx   # componente
```

El segundo necesita en `devDependencies`: `vitest`, `jsdom`, `@testing-library/react`, `@testing-library/user-event`
y `@testing-library/jest-dom`.

## 12. Reentrenar el modelo

Se hace en Google Colab con **`entrenar_colab.ipynb`** (está en este mismo directorio):

1. Abrir el cuaderno en Colab y cambiar el entorno a GPU (opcional, también corre en CPU).
2. Subir `apps/candidatos/ia/ner_documentos.py`.
3. (Opcional) subir un JSONL con hojas de vida reales anotadas.
4. Ejecutar el entrenamiento y descargar `modelo.pt`.
5. Copiarlo a `apps/candidatos/ia/modelo/modelo.pt` y reiniciar el servidor.

Para ver qué lee y qué marca el modelo en un archivo concreto, sin pasar por la API:

```bash
python apps/candidatos/ia/ner_documentos.py diagnostico --model apps/candidatos/ia/modelo/modelo.pt cv.pdf
```

## 13. Problemas frecuentes

| Síntoma | Causa probable | Qué hacer |
|---|---|---|
| La pantalla dice *«La importación con IA no está disponible»* | Falta `modelo.pt`, `torch` o `pytorch-crf` | Leer el motivo que muestra; instalar la dependencia o copiar el modelo |
| Una imagen o PDF escaneado sale con error de OCR | `easyocr` no está instalado | `pip install easyocr` (la primera vez descarga sus modelos, necesita internet) |
| Un archivo sale con *«No se encontró un correo electrónico»* | El CV no trae correo, o el OCR lo leyó mal | Corregir el archivo o cargar ese candidato por Excel |
| *«Ya existe un candidato con el correo…»* | Ya está cargado en esa compañía | Es el comportamiento esperado; no se duplica |
| El análisis tarda mucho la primera vez | Se carga el modelo y/o el OCR | Después queda en memoria; repetir el mismo archivo es inmediato (caché) |
| *«No se pudo conectar con el servidor»* a mitad de una importación | Se cortó la conexión | Revisar en **Candidatos** qué alcanzó a guardarse antes de reintentar |
| Nombre mal separado (p. ej. apellido compuesto) | Limitación del modelo | Agregar ejemplos reales anotados y reentrenar (sección 12) |